In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# 1. 하수관거 및 부대시설 데이터 불러오기
sewer = pd.read_csv("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/하수도관련/하수관거_20260527133919.csv")
facilities = pd.read_csv("C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/하수도관련/하수도+및+부대시설+현황_20260527133737.csv")

In [3]:
# 하수관거, 부대시설 파일에는 2023년 단일 데이터만 있으므로 2023년 데이터 활용

# 2. 필요한 컬럼 추출
s2 = sewer[['자치구별(2)', '2023.1', '2023.17']].copy()
s2.columns = ['자치구', 'SL_district', 'SF_district']
s2 = s2[~s2['자치구'].isin(['자치구별(2)', '소계'])]

In [4]:
fac = facilities[['자치구별(2)', '2023.5', '2023.6']].copy()
fac.columns = ['자치구', 'Manholes', 'CatchBasins']
fac = fac[~fac['자치구'].isin(['자치구별(2)', '소계'])]

In [5]:
# 결측치('-')를 0으로 치환하고 실수형으로 변환
for col in ['SL_district', 'SF_district']:
    s2[col] = s2[col].astype(str).str.replace(',', '').replace('-', '0').astype(float)
    
for col in ['Manholes', 'CatchBasins']:
    fac[col] = fac[col].astype(str).str.replace(',', '').replace('-', '0').astype(float)

In [6]:
# 자치구 기준으로 데이터프레임 병합
df = pd.merge(s2, fac, on='자치구', how='inner')

In [7]:
df.head()

,자치구,SL_district,SF_district,Manholes,CatchBasins
0,종로구,363920.0,0.0,10126.0,16383.0
1,중구,272027.0,0.0,8304.0,15045.0
2,용산구,375865.0,0.0,9970.0,17854.0
3,성동구,307722.0,355.0,8694.0,19332.0
4,광진구,370605.0,138.0,10693.0,25505.0


In [8]:
#  자치구 용량(C_gu) 연산
df['C_gu'] = df['SL_district'] * (1 + (df['SF_district'] / df['SL_district']).fillna(0))

In [9]:
# 관거 1m당 미세 배수 인프라 밀도 연산
df['MicroDrain_Density'] = (df['Manholes'] + df['CatchBasins']) / df['SL_district']

In [10]:
# 인프라 용량과 밀도가 '낮을수록' 위험(1.0)하고, '높을수록' 안전(0.1)하도록 정규화
def custom_reverse_scale(series, a=0.1, b=1.0):
    s_min = series.min()
    s_max = series.max()
    return a + ((s_max - series) * (b - a)) / (s_max - s_min)

# 하수처리 미흡 지표(S2) 및 배수 취약성 지표(S3) 산출
df['S2_Vulnerability'] = custom_reverse_scale(df['C_gu'])
df['S3_Vulnerability'] = custom_reverse_scale(df['MicroDrain_Density'])

In [12]:
df[['자치구', 'C_gu', 'MicroDrain_Density', 'S2_Vulnerability', 'S3_Vulnerability']]

,자치구,C_gu,MicroDrain_Density,S2_Vulnerability,S3_Vulnerability
0,종로구,363920.0,0.072843,0.809481,0.711083
1,중구,272027.0,0.085833,0.983052,0.391376
2,용산구,375865.0,0.074027,0.786919,0.681952
3,성동구,308077.0,0.091076,0.914959,0.262358
4,광진구,370743.0,0.097673,0.796594,0.100000
5,동대문구,404995.0,0.088448,0.731897,0.327028
6,중랑구,432510.0,0.088907,0.679926,0.315728
7,성북구,506058.0,0.083212,0.541006,0.455896
8,강북구,356780.0,0.083603,0.822967,0.446260
9,도봉구,329958.0,0.079387,0.873630,0.550021


### Column 명
- C_gu : 하수처리 용량 원시값
    - 자치구별 하수관거 전체 길이(SL)와 우수관거(빗물관, SF)의 비율
    - 절대적인 하수처리 인프라 용량
    - 숫자가 클수록 빗물을 감당할 파이프라인이 튼튼한 것
- MicroDrain_Density : 배수 인프라 밀도 
    - 하수관거 1m 당 빗물받이와 맨홀의 개수
    - 수치가 클수록 개수가 많은 것
- S2_Vulnerability : 하수처리 미흡 지표 (산출식에서 S2에 해당)
    - C_gu를 0과 1 사이로 변환한 지표
    - 1에 가까울수록 하수처리가 부실하여 홍수에 취약함을 나타냄
- S3_Vulnerability : 배수 취약성
    - MicroDrain_Density를 변환한 값
    - 1에 가까울수록 빗물받이와 맨홀이 부족한 것

In [17]:
# 수치가 높은 자치구
# 2. S2_Vulnerability 기준 내림차순 정렬 및 출력

s2_sorted = df.sort_values(by='S2_Vulnerability', ascending=False).reset_index(drop=True)
print("\n[하수처리 미흡 지표(S2) 취약 순위 (최솟값 0.1 보정)]")
display(s2_sorted[['자치구', 'C_gu', 'S2_Vulnerability']])



[하수처리 미흡 지표(S2) 취약 순위 (최솟값 0.1 보정)]


,자치구,C_gu,S2_Vulnerability
0,금천구,263054.0,1.000000
1,중구,272027.0,0.983052
2,성동구,308077.0,0.914959
3,도봉구,329958.0,0.873630
4,동작구,333210.0,0.867487
5,서대문구,348685.0,0.838257
6,강북구,356780.0,0.822967
7,종로구,363920.0,0.809481
8,광진구,370743.0,0.796594
9,용산구,375865.0,0.786919


In [18]:
# 3. S3_Vulnerability 기준 내림차순 정렬 및 출력

s3_sorted = df.sort_values(by='S3_Vulnerability', ascending=False).reset_index(drop=True)
print("\n[배수 취약성 지표(S3) 취약 순위 (최솟값 0.1 보정)]")
display(s3_sorted[['자치구', 'MicroDrain_Density', 'S3_Vulnerability']])


[배수 취약성 지표(S3) 취약 순위 (최솟값 0.1 보정)]


,자치구,MicroDrain_Density,S3_Vulnerability
0,송파구,0.061104,1.000000
1,노원구,0.064649,0.912731
2,영등포구,0.065647,0.888185
3,강서구,0.068214,0.825016
4,구로구,0.068215,0.824992
5,서대문구,0.068767,0.811388
6,강동구,0.069336,0.797382
7,강남구,0.069490,0.793594
8,종로구,0.072843,0.711083
9,용산구,0.074027,0.681952


### 1. 하수도 인프라 방어 지표 산출
- 빗물을 감당하는 관의 크기와 유입구의 수를 홍수 방어 지표로 연산
    - 하수처리 용량(C_gu) : 자치구별 하수관거 전체 길이와 우수관거(빗물관) 비율을 곱하여 산출한 절대적인 하수처리 용량
    - 배수 인프라 밀도(MicroDrain_Density) : 하수관거 1m 당 설치된 맨홀과 빗물받이의 총 개수

### 2. 위치 기반 매핑 및 데이터 병합
- 데이터에 포함된 자치구 기준으로 데이터 병합
- 결측치 0으로 치환, 문자열->실수형으로 변환하는 정제 과정 수행

### 3. 취약성 평가 지수
- 인프라 용량과 밀도가 낮을수록 홍수에 위험하므로, 작을수록 1에 가까워지도록 역정규화 수행
    - 하수처리 미흡 지표(S2) : 하수처리 용량 변환값, 1에 가까울수록 하수처리 부실
    - 배수 취약성 지표(S3) : 배수 인프라 밀도 변환값, 1에 가까울수록 부족함을 의미